In [ ]:
! pip install --upgrade langchain langchain-openai langchain-core langchain_community docx2txt pypdf langchain_chroma sentence_transformers

In [ ]:

import langchain
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


from langchain_openai import ChatOpenAI, OpenAIEmbeddings


from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter


from langchain_chroma import Chroma


import docx2txt
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer


In [ ]:
import langchain
print(langchain.__version__)

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGCHAIN_PROJECT", "RAG PROJECT")

print("OpenAI API Key configured:", bool(os.getenv("OPENAI_API_KEY")))
print("LangSmith Tracing configured:", bool(os.getenv("LANGCHAIN_API_KEY")))


Calling the llm


In [ ]:
import langchain_openai
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_response = llm.invoke("Hello! In one sentence, what is LangChain?") # verify LLM connectivity

print(llm_response.content)


Parsing O/P

In [ ]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
output_parser.invoke(llm_response)

Simple chain


In [ ]:
chain = llm | output_parser
result = chain.invoke("Explain Retrieval-Augmented Generation (RAG) in one sentence.")
print(result)


Structured 0utput

In [ ]:
from typing import List
from pydantic import BaseModel, Field

class MobileReview(BaseModel):
        
    phone_model: str = Field(description="Name and model of the phone")
    rating: float = Field(description="Overall rating out of 5")
    pros: List[str] = Field(description="List of positive aspects")
    cons: List[str] = Field(description="List of negative aspects")
    summary: str = Field(description="Brief summary of the review")

review_text = """
Just got my hands on the new Galaxy S21 and wow, this thing is slick! The screen is gorgeous,
colors pop like crazy. Camera's insane too, especially at night - my Insta game's never been
stronger. Battery life's solid, lasts me all day no problem.
Not gonna lie though, it's pretty pricey. And what's with ditching the charger? C'mon Samsung.
Also, still getting used to the new button layout, keep hitting Bixby by mistake.
Overall, I'd say it's a solid 4 out of 5. Great phone, but a few annoying quirks keep it from
being perfect. If you're due for an upgrade, definitely worth checking out!
"""

structured_llm = llm.with_structured_output(MobileReview)
output = structured_llm.invoke(review_text)
print(output)


Prompt template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("Tell me a concise and funny joke about {topic}.")
prompt.invoke({"topic": "software bugs"})


In [ ]:
chain = prompt | llm | output_parser
result = chain.invoke({"topic": "machine learning"})
print(result)


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Define prompt
prompt = ChatPromptTemplate.from_template("Provide 3 best practices for deploying {technology}.")

# 2. Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 3. Define output parser
output_parser = StrOutputParser()

# 4. Compose LCEL chain
chain = prompt | llm | output_parser

# 5. Invoke chain
result = chain.invoke({"technology": "FastAPI and LangChain"})
print(result)


LLM Messages

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

system_message = SystemMessage(content="You are a senior AI architect. Answer concisely with bullet points.")
human_message = HumanMessage(content="What are the three core stages of a RAG pipeline?")
response = llm.invoke([system_message, human_message])
print(response.content)


In [ ]:
template = ChatPromptTemplate([
    ("system", "You are an expert tutor. Provide an intuitive analogy for {topic}."),
    ("human", "Explain it simply so a beginner can understand.")
])

prompt_value = template.invoke({"topic": "Vector Embeddings"})
prompt_value


In [ ]:
llm.invoke(prompt_value)

In [ ]:
! pip install docx2txt pypdf unstructured

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from typing import List
from langchain_core.documents import Document

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap=200,
    length_function = len
)

docx_loader = Docx2txtLoader("sample_documents/cloud_architecture.docx")
documents = docx_loader.load()

splits = text_splitter.split_documents(documents)
print(f"Loaded {len(documents)} document, split into {len(splits)} chunks.")


In [ ]:
import os

def load_documents(folder_path: str) -> List[Document]:
    documents = []
    for filename in sorted(os.listdir(folder_path)):
        file_path = os.path.join(folder_path, filename)
        if filename.endswith('.pdf'):
            loader = PyPDFLoader(file_path)
        elif filename.endswith('.docx'):
            loader = Docx2txtLoader(file_path)
        else:
            print(f"Unsupported file type: {filename}")
            continue
        documents.extend(loader.load())
    return documents

folder_path = "sample_documents"
documents = load_documents(folder_path)
print(f"Loaded {len(documents)} documents from '{folder_path}'.")
splits = text_splitter.split_documents(documents)
print(f"Split into {len(splits)} total chunks across all documents.")


In [ ]:
embeddings = OpenAIEmbeddings()

###Embedding Documents

documents_embeddings= embeddings.embed_documents([split.page_content for split in splits])

print(f"Created Embeddings for {len(documents_embeddings)} document chunks.")

documents_embeddings[0]


In [ ]:
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

embeddings_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
document_embeddings = embeddings_function.embed_documents([split.page_content for split in splits])
print(f"Generated {len(document_embeddings)} sentence-transformer embeddings (dim={len(document_embeddings[0])}).")
print("First 5 dimensions of first chunk:", document_embeddings[0][:5])


In [ ]:
! pip install langchain_chroma -q

In [ ]:
#import shutil
##shutil.rmtree('chroma_db')

Create and persist Chroma Vector Store

In [ ]:
from langchain_chroma import Chroma

# Select embedding model:
# Option A (Local & Free, no API quota needed):
embeddings_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Option B (OpenAI, requires active OpenAI credits):
# embeddings_function = OpenAIEmbeddings()

collection_name = "my_collection"

vectorstore = Chroma.from_documents(
    collection_name=collection_name,
    documents=splits,
    embedding=embeddings_function,
    persist_directory="./chroma_db"
)

print("Vector Store created and persisted to './chroma_db'")


In [ ]:
# performing similarity search

query = "What is the policy for remote work and home office stipend?"
search_results = vectorstore.similarity_search(query, k=2)

print(f"\nTop 2 most relevant chunks for query: '{query}'\n")
for i, result in enumerate(search_results, 1):
    print(f"Result {i}:")
    print(f"Source: {result.metadata.get('source', 'Unknown')}")
    print(f"Content:\n{result.page_content}\n")


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
retriever_docs = retriever.invoke("What vector database and chunk size are recommended?")
print(f"Retrieved {len(retriever_docs)} chunks.")
print(retriever_docs[0].page_content[:200])


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer: """

prompt = ChatPromptTemplate.from_template(template)


In [ ]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()} | prompt
)
rag_chain.invoke("What is the home office stipend amount?")


In [ ]:
def doc2str(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain = (
    {"context": retriever | doc2str, "question": RunnablePassthrough()} | prompt
)
rag_chain.invoke("What is the home office stipend amount?")


In [ ]:
rag_chain = (
    {"context": retriever | doc2str, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

question = "What is the policy for remote work and equipment allowances?"
response = rag_chain.invoke(question)
print(response)


<h1> Conversational RAG </h1>

<h5> HAndling Follow up Questions



In [ ]:
# Example conversation

from langchain_core.messages import HumanMessage, AIMessage

chat_history = []
chat_history.extend([
    HumanMessage(content=question),
    AIMessage(content=response)
])

In [ ]:
chat_history

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return as it is."
)
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

contextualize_chain = contextualize_q_prompt | llm | StrOutputParser()
standalone_q = contextualize_chain.invoke({"input": "How much stipend do I get for it?", "chat_history": chat_history})
print("Reformulated standalone question:", standalone_q)


In [ ]:
from langchain_classic.chains import create_history_aware_retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
retrieved_docs = history_aware_retriever.invoke({"input": "How much allowance is provided for it?", "chat_history": chat_history})
print(f"Retrieved {len(retrieved_docs)} docs for contextual question.")


In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI Assistant. Use the following context to answer the user's question."),
    # ("system", "Tell me a joke on Programming"),
    ("system", "Context: {context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human","{input}")
])

question_answer_chain = create_stuff_documents_chain(llm , qa_prompt)

rag_chain= create_retrieval_chain(history_aware_retriever, question_answer_chain)


In [ ]:
conv_response = rag_chain.invoke({"input": "Can I work from home and what are the core hours?", "chat_history": chat_history})
print(conv_response['answer'])


<h1> Multi User Chatbot

In [ ]:
import sqlite3
from datetime import datetime
from langchain_core.messages import HumanMessage, AIMessage

DB_NAME = "rag_app.db"

def get_db_connection():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    return conn

def create_application_logs():
    conn = get_db_connection()
    conn.execute('''CREATE TABLE IF NOT EXISTS application_logs
                    (id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT,
                    user_query TEXT,
                    gpt_response TEXT,
                    model TEXT,
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
    conn.commit()
    conn.close()

def insert_application_logs(session_id, user_query, gpt_response, model):
    conn = get_db_connection()
    conn.execute('INSERT INTO application_logs (session_id, user_query, gpt_response, model) VALUES (?,?,?,?)',
                 (session_id, user_query, gpt_response, model))
    conn.commit()
    conn.close()

def get_chat_history(session_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    # Fix: ensure (session_id,) is a tuple
    cursor.execute('SELECT user_query, gpt_response FROM application_logs WHERE session_id = ? ORDER BY created_at', (session_id,))
    messages = []
    for row in cursor.fetchall():
        messages.extend([
            HumanMessage(content=row['user_query']),
            AIMessage(content=row['gpt_response'])
        ])
    conn.close()
    return messages

# Initialize the database
create_application_logs()
print("SQLite database and 'application_logs' table ready.")


In [ ]:
import uuid
session_id = str(uuid.uuid4())

chat_history = get_chat_history(session_id)
print(f"Initial chat history length: {len(chat_history)}")

question1 = "What is the remote work policy and equipment stipend at Acme Corp?"
answer1 = rag_chain.invoke({"input": question1, "chat_history": chat_history})['answer']

insert_application_logs(session_id, question1, answer1, "gpt-4o-mini")
print(f"Human: {question1}")
print(f"AI: {answer1}\n")


In [ ]:
question2 = "How many days of annual leave do I get, and what is the policy for sick leave?"
chat_history = get_chat_history(session_id)
print(f"Retrieved chat history length: {len(chat_history)} messages")

answer2 = rag_chain.invoke({"input": question2, "chat_history": chat_history})['answer']
insert_application_logs(session_id, question2, answer2, "gpt-4o-mini")
print(f"Human: {question2}")
print(f"AI: {answer2}\n")


In [ ]:
# Verify persisted logs in SQLite
conn = get_db_connection()
rows = conn.execute('SELECT id, session_id, user_query, gpt_response, model, created_at FROM application_logs').fetchall()
print(f"Total persisted log records: {len(rows)}\n")
for r in rows:
    print(f"[{r['created_at']}] Session {r['session_id'][:8]}... | Model: {r['model']}")
    print(f"  User: {r['user_query']}")
    print(f"  AI:   {r['gpt_response'][:120]}...\n")
conn.close()
